# QC — histograms

`participants_qc.csv` for every cohort that has one, one row per (subject,
task). Measurements only; no threshold is applied anywhere here.

`fd_source` is a string (which column FD was read from), so it is not in the
grid — `pd.crosstab(qc["cohort"], qc["fd_source"])` if you want it.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import nbtools as nb

METRICS = ["mean_fd", "frac_good_frames", "frac_parcels_empty",
           "frac_stimulus_covered", "peak_isc", "best_lag_tr"]

paths = sorted((nb.output_root() / "meta" / "cohorts").glob("cohort=*/participants_qc.csv"))
qc = pd.concat([pd.read_csv(p, dtype={"sub": str}) for p in paths], ignore_index=True)
cohorts = sorted(qc["cohort"].unique())
print(f"{len(qc)} row(s), {len(cohorts)} cohort(s): {cohorts}")

# A blank panel below means one of these, not a plotting bug.
absent = [m for m in METRICS if m not in qc.columns]
empty = [m for m in METRICS if m in qc.columns and qc[m].isna().all()]
if absent:
    print(f"not a column in participants_qc.csv: {absent}")
if empty:
    print(f"present but all-NaN (missing input, e.g. ISC needs >=3 subjects/task): {empty}")

In [ ]:
fig, axes = plt.subplots(len(METRICS), len(cohorts),
                         figsize=(3.1 * len(cohorts), 1.9 * len(METRICS)))
axes = np.atleast_2d(axes).reshape(len(METRICS), len(cohorts))

for i, metric in enumerate(METRICS):
    for j, cohort in enumerate(cohorts):
        ax = axes[i, j]
        values = (qc.loc[qc["cohort"] == cohort, metric].dropna()
                  if metric in qc.columns else pd.Series(dtype=float))
        if len(values):
            ax.hist(values, bins=25)
            ax.set_xlabel(f"n={len(values)}  med={values.median():.3g}", fontsize=7)
        else:
            ax.text(0.5, 0.5, "no data", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8, color="grey")
        if i == 0:
            ax.set_title(cohort, fontsize=9)
        if j == 0:
            ax.set_ylabel(metric, fontsize=8)
        ax.tick_params(labelsize=7)

fig.tight_layout()